In [ ]:
import os
import json
import time
import difflib
import multiprocessing
from datetime import datetime
from pathlib import Path
import re
from typing import List, Dict, Tuple, Optional, Any
from dotenv import load_dotenv
import functools
from tqdm import tqdm
from tenacity import retry, stop_after_attempt, wait_exponential, retry_if_exception_type

# LangChain imports
from langchain_openai import ChatOpenAI
from langchain_anthropic import ChatAnthropic
from langchain_deepseek import ChatDeepSeek
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_mistralai.chat_models import ChatMistralAI
from langchain_xai import ChatXAI
from langchain.schema.messages import SystemMessage, HumanMessage

# Configuration
config = {
    "path_candidates": "./localization_combination_results/union/selected_results/union_top_3_results.json", 
    "llm": "deepseek",  # provider name (chatgpt, claude, deepseek, grok, gemini, mistral)
    "model_name": "deepseek-chat",  # exact model name that points to a certain model
    "temperature": 0,
    "num_processes": 16, 
    "repair_cache": False,
    "update_repair_cache": True,
    "nonfix_cache": False, 
    "update_nonfix_cache": True, 
    "max_tokens": 8192,
    "save_cost": True  # if True, only process ground truth files and ignore others
}

def load_llm(llm_provider: str, model_name: str, temperature: float, max_tokens: int):
    """Load the specified LLM model."""
    load_dotenv()  # Load environment variables from .env file
    
    if llm_provider == "chatgpt":
        return ChatOpenAI(
            api_key=os.getenv("OPENAI_API_KEY"),
            model_name=model_name,
            temperature=temperature,
            max_tokens=max_tokens
        )
    elif llm_provider == "claude":
        return ChatAnthropic(
            api_key=os.getenv("ANTHROPIC_API_KEY"),
            model_name=model_name,
            temperature=temperature,
            max_tokens=max_tokens
        )
    elif llm_provider == "deepseek":
        return ChatDeepSeek(
            api_key=os.getenv("DEEPSEEK_API_KEY"),
            model_name=model_name,
            temperature=temperature,
            max_tokens=max_tokens
        )
    elif llm_provider == "gemini":
        return ChatGoogleGenerativeAI(
            api_key=os.getenv("GOOGLE_API_KEY"),
            model_name=model_name,
            temperature=temperature,
            max_tokens=max_tokens
        )
    elif llm_provider == "mistral":
        return ChatMistralAI(
            api_key=os.getenv("MISTRAL_API_KEY"),
            model_name=model_name,
            temperature=temperature,
            max_tokens=max_tokens
        )
    elif llm_provider == "grok":
        return ChatXAI(
            api_key=os.getenv("XAI_API_KEY"),
            model_name=model_name,
            temperature=temperature,
            max_tokens=max_tokens
        )
    else:
        raise ValueError(f"Unsupported LLM provider: {llm_provider}")

def check_truncation(response, llm_provider: str, max_tokens: int) -> bool:
    """
    Attempts to determine if the LLM response was truncated due to token limits.
    Adapts to different response metadata formats based on provider.
    """
    # Different providers use different metadata structures and keys
    if llm_provider == "chatgpt" or llm_provider == "deepseek":
        # OpenAI usually has finish_reason in response metadata
        if hasattr(response, 'response_metadata') and 'finish_reason' in response.response_metadata:
            return response.response_metadata['finish_reason'] == 'length'
        # Alternative location for OpenAI
        elif hasattr(response, 'additional_kwargs') and 'finish_reason' in response.additional_kwargs:
            return response.additional_kwargs['finish_reason'] == 'length'
    
    elif llm_provider == "claude":
        # Anthropic may use stop_reason or similar
        if hasattr(response, 'response_metadata') and 'stop_reason' in response.response_metadata:
            return response.response_metadata['stop_reason'] == 'max_tokens'
        elif hasattr(response, 'additional_kwargs') and 'stop_reason' in response.additional_kwargs:
            return response.additional_kwargs['stop_reason'] == 'max_tokens'
    
    elif llm_provider == "gemini":
        # Google may use different terminology
        if hasattr(response, 'response_metadata') and 'finish_reason' in response.response_metadata:
            return response.response_metadata['finish_reason'] == 'MAX_TOKENS'
        elif hasattr(response, 'additional_kwargs') and 'finish_reason' in response.additional_kwargs:
            return response.additional_kwargs['finish_reason'] == 'MAX_TOKENS'
    
    elif llm_provider == "mistral":
        # Mistral terminology
        if hasattr(response, 'response_metadata') and 'finish_reason' in response.response_metadata:
            return response.response_metadata['finish_reason'] == 'length'
        elif hasattr(response, 'additional_kwargs') and 'finish_reason' in response.additional_kwargs:
            return response.additional_kwargs['finish_reason'] == 'length'
    
    # For other providers or fallback: use heuristic checks
    
    # Check 1: Look for incomplete tags as a sign of truncation
    content = response.content if hasattr(response, 'content') else str(response)
    incomplete_tags = (
        "<original_code" in content and "</original_code" not in content or
        "<fixed_code" in content and "</fixed_code" not in content
    )
    if incomplete_tags:
        print(f"Warning: Response appears truncated based on incomplete tags.")
        return True
    
    # Check 2: Check if the content almost exactly matches the max_tokens
    # This is a heuristic that might indicate truncation
    token_estimate = len(content.split()) * 1.3  # Rough token estimate (words * 1.3)
    if token_estimate > max_tokens * 0.95:
        print(f"Warning: Response may be truncated (estimated tokens: {token_estimate}, max: {max_tokens})")
        return True
    
    print(f"No truncation detected for {llm_provider} response.")
    return False

def create_prompt(file_path: str, problem_statement: str, file_content: str, line_numbers: Optional[int] = None):
    """Create prompts for asking LLM to repair a buggy file."""
    line_numbers = 10

    system_prompt = """
You are an expert software engineer tasked with fixing bugs in a given code file. Your primary goal is to carefully analyze the code, identify any bugs, and provide fixes that will effectively resolve the issues.
"""
    
    user_prompt = f"""

Here's the context and information you need to work with:

File content:
<file_content>
{file_content}
</file_content>

File path:
<file_path>
{file_path}
</file_path>

Problem statement:
<problem_statement>
{problem_statement}
</problem_statement>

Maximum lines per section:
<max_lines_per_section>
{line_numbers}
</max_lines_per_section>

Instructions:

1. Analyze the code thoroughly, keeping in mind the problem statement.

2. Follow this step-by-step process:
   a. Read through the entire code file
   b. Identify potential bugs or issues
   c. For each potential bug, explain why it's problematic
   d. Develop a fix for each identified bug
   e. Verify that the fix resolves the issue without introducing new problems

3. After your code review, provide the original and fixed code sections using the following format:

   <original_code_n>
   [Exact copy of the nth problematic code section]
   </original_code_n>

   <fixed_code_n>
   [Fixed version of the nth code section]
   </fixed_code_n>

   Replace 'n' with sequential numbers starting from 0 (e.g., original_code_0, fixed_code_0, original_code_1, fixed_code_1, etc.).

4. Important guidelines:
   - Only include sections that have been changed in your final output
   - Ensure that each fixed code section actually modifies something from the original
   - Ensure that each problematic section does not exceed the number of lines specified in <max_lines_per_section>
   - If no bugs are found in the entire code, state "No issues were identified in the code."

Example output structure (do not copy this content, it's just to illustrate the format):

<original_code_0>
def calculate_sum(a, b)
    a - b
</original_code_0>

<fixed_code_0>
def calculate_sum(a, b)
    a + b
</fixed_code_0>


Please proceed with your analysis and provide the necessary fixes for the given code file. Your final output should consist only of the original and fixed code sections and should not duplicate or rehash any of the work you did in the code review process.
"""
    
    return system_prompt, user_prompt

def extract_fixes_from_response(response: str) -> List[Tuple[str, str]]:
    """Extract original and fixed code pairs from LLM response."""
    original_pattern = r"<original_code_\d+>(.*?)</original_code_\d+>"
    fixed_pattern = r"<fixed_code_\d+>(.*?)</fixed_code_\d+>"
    
    # Use re.DOTALL to match across line breaks
    original_sections = re.findall(original_pattern, response, re.DOTALL)
    fixed_sections = re.findall(fixed_pattern, response, re.DOTALL)
    
    # Ensure we have matching pairs
    pairs = []
    for i in range(min(len(original_sections), len(fixed_sections))):
        # Strip leading and trailing whitespace but preserve internal formatting
        original = original_sections[i].strip()
        fixed = fixed_sections[i].strip()
        pairs.append((original, fixed))
    
    return pairs

def create_unified_diff(original_content: str, fixes: List[Tuple[str, str]], file_path: str) -> str:
    """Generate a unified diff from original content and fixes."""
    # Create a copy of the original content
    new_content = original_content
    
    # Apply all fixes
    for original, fixed in fixes:
        new_content = new_content.replace(original, fixed)
    
    # Generate unified diff
    original_lines = original_content.splitlines(True)
    new_lines = new_content.splitlines(True)
    
    diff = difflib.unified_diff(
        original_lines,
        new_lines,
        fromfile=f'a/{file_path}',
        tofile=f'b/{file_path}',
        n=3  # Context lines
    )
    
    return ''.join(diff)

@retry(
    stop=stop_after_attempt(5),
    wait=wait_exponential(multiplier=1, min=60, max=600),
    retry=retry_if_exception_type((ConnectionError, TimeoutError))
)
def query_llm(llm_provider: str, model_name: str, temperature: float, max_tokens: int, 
             system_prompt: str, user_prompt: str) -> Tuple[Any, str]:
    """Query the LLM with retry logic for connection errors."""
    # Load a new instance of the LLM
    llm = load_llm(llm_provider, model_name, temperature, max_tokens)
    
    # For deepseek, combine system and user prompts as it doesn't support system messages
    if llm_provider == "deepseek":
        messages = [HumanMessage(content=f"{system_prompt}\n\n{user_prompt}")]
    else:
        messages = [SystemMessage(content=system_prompt), HumanMessage(content=user_prompt)]
    
    # Invoke the LLM
    response = llm.invoke(messages)
    return response, response.content

def query_llm_with_retry(llm_provider: str, model_name: str, temperature: float, max_tokens: int,
                        system_prompt: str, user_prompt: str, instance_id: str, file_path: str, 
                        file_content: str, problem_statement: str, output_dir: Path, is_ground_truth: bool) -> Tuple[str, str]:
    """Query the LLM with retry logic for token limits."""
    # Initial query without line number constraints
    response, raw_response = query_llm(
        llm_provider=llm_provider,
        model_name=model_name,
        temperature=temperature,
        max_tokens=max_tokens,
        system_prompt=system_prompt,
        user_prompt=user_prompt
    )
    
    # Save raw response
    category = "ground_truth" if is_ground_truth else "nonfix"
    output_path = output_dir / "raw_output" / instance_id / category
    output_path.mkdir(parents=True, exist_ok=True)
    with open(output_path / "response.txt", "w", encoding="utf-8") as f:
        f.write(raw_response)
    
    # Check if response was truncated using provider-specific metadata
    is_truncated = check_truncation(response, llm_provider, max_tokens)
    
    # Check if we got complete response with at least one pair of original/fixed
    fixes = extract_fixes_from_response(raw_response)
    
    # If the response is truncated or no fixes found, retry with decreasing line constraints
    if is_truncated or len(fixes) == 0:
        for line_limit in [5, 2]:
            print(f"Retrying with {line_limit} line limit for {instance_id}/{file_path}")
            
            # Create a more constrained prompt
            system_prompt_limited, user_prompt_limited = create_prompt(
                file_path=file_path,
                problem_statement=problem_statement,
                file_content=file_content,
                line_numbers=line_limit
            )
            
            # Retry with the constrained prompt
            response, raw_response = query_llm(
                llm_provider=llm_provider,
                model_name=model_name,
                temperature=temperature,
                max_tokens=max_tokens,
                system_prompt=system_prompt_limited,
                user_prompt=user_prompt_limited
            )
            
            # Save the constrained response
            with open(output_path / f"response_{line_limit}.txt", "w", encoding="utf-8") as f:
                f.write(raw_response)
            
            # Check if response is still truncated
            is_truncated = check_truncation(response, llm_provider, max_tokens)
            
            # Check if we have a complete response now
            fixes = extract_fixes_from_response(raw_response)
            if not is_truncated and len(fixes) > 0:
                break
    
    # Create unified diff from the fixes
    diff = create_unified_diff(file_content, fixes, file_path)
    
    return raw_response, diff

def process_instance(args: Tuple):
    """Process a single instance for multiprocessing.
    This function is now refactored to avoid passing the LLM instance directly."""
    
    # Unpack the arguments
    instance_id, file_path, is_ground_truth, ground_truth_paths, cache_data, config_data, output_dir = args
    
    try:
        llm_provider = config_data["llm"]
        model_name = config_data["model_name"]
        temperature = config_data["temperature"]
        max_tokens = config_data["max_tokens"]
        
        # Check cache first if enabled
        cache_key = "repair_cache" if is_ground_truth else "nonfix_cache"
        update_cache_key = "update_repair_cache" if is_ground_truth else "update_nonfix_cache"
        
        # Determine if we should use cache
        use_cache = config_data[cache_key]
        cache_exists = False
        
        # Check if cache exists and contains this instance+path
        if use_cache and instance_id in cache_data and file_path in cache_data[instance_id]:
            cache_exists = True
            cached_diff = cache_data[instance_id][file_path]
        
        # If we should use cache and it exists, return the cached result
        if use_cache and cache_exists:
            # Return early with cached diff
            return instance_id, file_path, is_ground_truth, cached_diff, ""
        
        # No cache or cache disabled, so query the LLM
        # Read the file content
        full_path = f"./codebases/{instance_id}/{file_path}"
        with open(full_path, "r", encoding="utf-8") as f:
            file_content = f.read()
        
        # Create the problem statement - a simple placeholder
        problem_statement = f"There is a bug in the file {file_path} that needs fixing."
        
        # Create prompts
        system_prompt, user_prompt = create_prompt(
            file_path=file_path,
            problem_statement=problem_statement,
            file_content=file_content
        )
        
        # Query the LLM
        raw_response, diff = query_llm_with_retry(
            llm_provider=llm_provider,
            model_name=model_name,
            temperature=temperature,
            max_tokens=max_tokens,
            system_prompt=system_prompt,
            user_prompt=user_prompt,
            instance_id=instance_id,
            file_path=file_path,
            file_content=file_content,
            problem_statement=problem_statement,
            output_dir=output_dir,
            is_ground_truth=is_ground_truth
        )
        
        # Save the diff
        category = "ground_truth" if is_ground_truth else "nonfix"
        diff_path = output_dir / "diff" / instance_id / category
        diff_path.mkdir(parents=True, exist_ok=True)
        with open(diff_path / "patch.txt", "w", encoding="utf-8") as f:
            f.write(diff)
        
        # Return the results
        return instance_id, file_path, is_ground_truth, diff, raw_response
        
    except Exception as e:
        print(f"Error processing {instance_id}/{file_path}: {e}")
        return instance_id, file_path, is_ground_truth, "", f"ERROR: {str(e)}"


# Create timestamp for output directory
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
output_dir = Path(f"./repairing/{timestamp}")
output_dir.mkdir(parents=True, exist_ok=True)

# Save the config
with open(output_dir / "config.json", "w") as f:
    json.dump(config, f, indent=2)

# Load the ground truth buggy file paths
with open("./ground_truth/bug_paths.json", "r") as f:
    ground_truth_paths = json.load(f)

# Load the candidate file paths
with open(config["path_candidates"], "r") as f:
    candidate_paths = json.load(f)

# Initialize cache data
repair_cache = {}
nonfix_cache = {}

# Load cache if enabled
if config["repair_cache"]:
    cache_path = f"./repairing/cache/{config['llm']}/{config['model_name']}/repair_history.json"
    if os.path.exists(cache_path):
        with open(cache_path, "r") as f:
            repair_cache = json.load(f)

if config["nonfix_cache"]:
    cache_path = f"./repairing/cache/{config['llm']}/{config['model_name']}/nonfix_history.json"
    if os.path.exists(cache_path):
        with open(cache_path, "r") as f:
            nonfix_cache = json.load(f)

# Build tasks for multiprocessing
tasks = []

# For each instance, process the ground truth file path and candidates
for instance_id, candidates in candidate_paths.items():
    if instance_id not in ground_truth_paths:
        print(f"Warning: No ground truth for instance {instance_id}")
        continue
    
    # Get the ground truth file path
    ground_truth_path = ground_truth_paths[instance_id]
    
    # Check if we should process the ground truth (not in cache or cache disabled)
    use_repair_cache = config["repair_cache"]
    repair_cache_exists = (
        instance_id in repair_cache and 
        ground_truth_path in repair_cache[instance_id]
    )
    
    if not (use_repair_cache and repair_cache_exists):
        # Add ground truth task - note we don't pass the LLM instance
        tasks.append((
            instance_id, ground_truth_path, True, ground_truth_paths,
            repair_cache, config, output_dir
        ))
    
    # Skip non-ground truth files if save_cost is enabled
    if config["save_cost"]:
        continue
    
    # Process candidates that are not the ground truth
    for candidate_path in candidates:
        if candidate_path == ground_truth_path:
            continue  # Skip ground truth, we've handled it separately
        
        # Check if we should process this candidate (not in cache or cache disabled)
        use_nonfix_cache = config["nonfix_cache"]
        nonfix_cache_exists = (
            instance_id in nonfix_cache and 
            candidate_path in nonfix_cache[instance_id]
        )
        
        if not (use_nonfix_cache and nonfix_cache_exists):
            # Add candidate task - note we don't pass the LLM instance
            tasks.append((
                instance_id, candidate_path, False, ground_truth_paths,
                nonfix_cache, config, output_dir
            ))

# Process tasks using multiprocessing with tqdm progress bar
results = []
with multiprocessing.Pool(processes=config["num_processes"]) as pool:
    for result in tqdm(pool.imap_unordered(process_instance, tasks), total=len(tasks), desc="Processing files"):
        results.append(result)

# Update caches if needed
if config["update_repair_cache"]:
    for instance_id, file_path, is_ground_truth, diff, _ in results:
        if is_ground_truth and diff:  # Only update for ground truth paths with actual diffs
            if instance_id not in repair_cache:
                repair_cache[instance_id] = {}
            repair_cache[instance_id][file_path] = diff
    
    # Save updated repair cache
    cache_dir = Path(f"./repairing/cache/{config['llm']}/{config['model_name']}")
    cache_dir.mkdir(parents=True, exist_ok=True)
    with open(cache_dir / "repair_history.json", "w") as f:
        json.dump(repair_cache, f, indent=2)

if config["update_nonfix_cache"]:
    for instance_id, file_path, is_ground_truth, diff, _ in results:
        if not is_ground_truth and diff:  # Only update for non-ground truth paths with actual diffs
            if instance_id not in nonfix_cache:
                nonfix_cache[instance_id] = {}
            nonfix_cache[instance_id][file_path] = diff
    
    # Save updated nonfix cache
    cache_dir = Path(f"./repairing/cache/{config['llm']}/{config['model_name']}")
    cache_dir.mkdir(parents=True, exist_ok=True)
    with open(cache_dir / "nonfix_history.json", "w") as f:
        json.dump(nonfix_cache, f, indent=2)

print(f"All tasks completed. Results saved to {output_dir}")

In [ ]:
import os
import json
from pathlib import Path
from typing import List, Dict, Any, Tuple

def build_patches_json_files(experiment_dir: Path, model_name: str) -> None:
    """
    Build three JSON files from the diffs generated during the experiment:
    1. ground_truth_patches.json - only ground truth diffs
    2. nonfix_patches.json - only nonfix diffs
    3. combined_patches.json - both ground truth and nonfix diffs combined
    
    Args:
        experiment_dir: Path to the experiment directory (containing the diffs)
        model_name: Name of the model used for the experiment
    """
    # These will store our patches
    ground_truth_patches = []
    nonfix_patches = []
    combined_patches = []
    
    # Locate the diff directory
    diff_dir = experiment_dir / "diff"
    
    # Make sure the diff directory exists
    if not diff_dir.exists():
        print(f"Error: Diff directory {diff_dir} does not exist")
        return
    
    # Process each instance directory
    for instance_dir in diff_dir.iterdir():
        if not instance_dir.is_dir():
            continue
            
        instance_id = instance_dir.name
        
        # Get ground truth diffs
        ground_truth_dir = instance_dir / "ground_truth"
        ground_truth_patch = ""
        if ground_truth_dir.exists() and ground_truth_dir.is_dir():
            patch_file = ground_truth_dir / "patch.txt"
            if patch_file.exists():
                with open(patch_file, "r", encoding="utf-8") as f:
                    ground_truth_patch = f.read()
                
                if ground_truth_patch.strip():
                    # Add to ground truth patches
                    ground_truth_patches.append({
                        "instance_id": instance_id,
                        "model_patch": ground_truth_patch,
                        "model_name_or_path": model_name
                    })
        
        # Get nonfix diffs
        nonfix_dir = instance_dir / "nonfix"
        nonfix_patch = ""
        if nonfix_dir.exists() and nonfix_dir.is_dir():
            patch_file = nonfix_dir / "patch.txt"
            if patch_file.exists():
                with open(patch_file, "r", encoding="utf-8") as f:
                    nonfix_patch = f.read()
                
                if nonfix_patch.strip():
                    # Add to nonfix patches
                    nonfix_patches.append({
                        "instance_id": instance_id,
                        "model_patch": nonfix_patch,
                        "model_name_or_path": model_name
                    })
        
        # Combine patches if both exist
        if ground_truth_patch.strip() or nonfix_patch.strip():
            combined_patch = combine_patches(ground_truth_patch, nonfix_patch)
            combined_patches.append({
                "instance_id": instance_id,
                "model_patch": combined_patch,
                "model_name_or_path": model_name
            })
    
    # Save the patches JSON files
    with open(experiment_dir / "ground_truth_patches.json", "w", encoding="utf-8") as f:
        json.dump(ground_truth_patches, f, indent=4)
    
    with open(experiment_dir / "nonfix_patches.json", "w", encoding="utf-8") as f:
        json.dump(nonfix_patches, f, indent=4)
    
    with open(experiment_dir / "combined_patches.json", "w", encoding="utf-8") as f:
        json.dump(combined_patches, f, indent=4)
    
    print(f"Created patches JSON files:")
    print(f"  - ground_truth_patches.json: {len(ground_truth_patches)} patches")
    print(f"  - nonfix_patches.json: {len(nonfix_patches)} patches")
    print(f"  - combined_patches.json: {len(combined_patches)} patches")

def combine_patches(ground_truth_patch: str, nonfix_patch: str) -> str:
    """
    Combine ground truth and nonfix patches into one unified diff.
    
    Args:
        ground_truth_patch: The ground truth patch content
        nonfix_patch: The nonfix patch content
        
    Returns:
        A combined patch in unified diff format
    """
    # If one patch is empty, return the other
    if not ground_truth_patch.strip():
        return nonfix_patch
    if not nonfix_patch.strip():
        return ground_truth_patch
    
    # Simply concatenate the patches with a separator
    combined = ground_truth_patch
    
    # Add a separator if needed
    if not combined.endswith("\n"):
        combined += "\n"
    
    combined += "\n# --- Nonfix patches below --- #\n\n"
    combined += nonfix_patch
    
    return combined

# Hardcoded directory path
experiment_dir = Path(f"./repairing/{timestamp}")

print(f"Processing experiment directory: {experiment_dir}")

# Check if the directory exists
if not experiment_dir.exists():
    print(f"Error: Experiment directory {experiment_dir} does not exist")
    raise

# Check if the directory has a config.json
config_file = experiment_dir / "config.json"
if not config_file.exists():
    print(f"Error: No config.json found in {experiment_dir}")
    raise

# Read the config to get the model name
with open(config_file, "r", encoding="utf-8") as f:
    config = json.load(f)

model_name = config.get("model_name", "unknown_model")

# Build the patches JSON files
build_patches_json_files(experiment_dir, model_name)


In [ ]:
python3 -m swebench.harness.run_evaluation --dataset_name princeton-nlp/SWE-bench_Lite --predictions_path /home/tweichuan/project/repairing/20250402_115334/ground_truth_patches.json --max_workers 4 --run_id deepseek_chat_test

In [ ]:
# from langchain_openai import ChatOpenAI
# from langchain_anthropic import ChatAnthropic
# from langchain_deepseek import ChatDeepSeek
# from langchain_google_genai import ChatGoogleGenerativeAI
# from langchain_mistralai.chat_models import ChatMistralAI
# from langchain_xai import ChatXAI

In [ ]:
# [
#     {
#         "instance_id": {instance_id},
#         "model_patch": {unified diff content},
#         "model_name_or_path": "model_name"
#     },
#     ...
# ]

In [ ]:
# ground_truth = "./ground_truth/bug_paths.json"

# config = {
#     "path_candidates": "candidates.json", 
#     "llm": "provider name (chatgpt, claude, deepseek, grok, gemini, mistral)", 
#     "model_name": "exact model name that point to certain model", 
#     "temperature": 0.8,
#     "num_processes": 8, 
#     "repair_cache": False,
#     "update_repair_cache": False,
#     "nonfix_cache": True, 
#     "update_nonfix_cache": True, 
#     "max_tokens": 8192
# }

In [ ]:
system_prompt = (
    "You are an expert software engineer. Your task is to fix a bug in the provided code. "
    "Analyze the code carefully and fix any bugs that you find. "
    "You must respond in a specific format, showing both the original problematic code sections "
    "and your fixed versions. Only include sections that need changes."
)